# Visual 1: Property Value Distribution Over Time (Austin vs. U.S.)

This chart shows how home prices in Austin compare to the rest of the United States over time. You can use the year slider to see how the distribution of home prices has changed. Austin has more expensive homes than the U.S. average, with a bigger share of homes costing over $300,000.

In [ ]:
%pip install pandas numpy matplotlib altair pathlib

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path

# Load data for Austin and U.S.
prop_val_path = Path('../data/austin_property_value.csv')
prop_val = pd.read_csv(prop_val_path)

# Normalize column names and keep only relevant columns
prop_val = prop_val.rename(columns={
    'Value Bucket': 'bucket',
    'Year': 'year',
    'Place': 'place',
    'share': 'share'
})

#### Import Altair Chart to JSON

In [ ]:
import altair as alt
import os

os.makedirs("../reports/exports", exist_ok=True)

# Embed all data in the JSON (standalone spec)
alt.data_transformers.enable("default", max_rows=None)

# Make it responsive in Panel (or set fixed pixels if you prefer)
#chart = chart.properties(width="container", height=360)

# Save as a Vega-Lite JSON spec
#chart.save("../reports/exports/PropertyDistribution_Visual.Altair.json")   # filename can be whatever; .json is fine
print("../reports/exports/PropertyDistribution_Visual.Altair.json")


## Visual V2 - Fewer Buckets to make it easier to digest

#### Import Property Value Data

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path

# Load data for Austin and U.S.
prop_val_path = Path('../data/austin_property_value.csv')
prop_val = pd.read_csv(prop_val_path)

#### Remove Unecessary Columns and Normalize names

In [ ]:
# Make a cleaned copy
prop_val = prop_val.drop(
    columns=["Value Bucket ID", "Property Value by Bucket Moe", "Place ID"]
).copy()

# Quick check
print(prop_val.head())
print(prop_val['Place'].unique())

In [ ]:
# Normalize column names and keep only relevant columns
prop_val = prop_val.rename(columns={
    'Value Bucket': 'bucket',
    'Year': 'year',
    'Place': 'place',
    'share': 'share',
    'Property Value by Bucket' : '#Properties'
})

In [ ]:
prop_val.head(5)

In [ ]:
prop_val.info()

In [ ]:
# Keep only Austin and United States
prop_val = prop_val[prop_val['place'].isin(['Austin, TX', 'United States'])].copy()

In [ ]:
# Create a simpler region column
prop_val['region'] = prop_val['place'].replace({'Austin, TX': 'Austin', 'United States': 'U.S.'})

In [ ]:
prop_val.head()

In [ ]:
prop_val.tail()

#### Ensure dtypes 

In [ ]:
# Ensure dtypes
prop_val['year'] = pd.to_numeric(prop_val['year'], errors='coerce').astype('Int64')
prop_val = prop_val.dropna(subset=['year'])
prop_val['year'] = prop_val['year'].astype(int)

In [ ]:
prop_val.head()

In [ ]:
prop_val.info()

### Map fine buckets into larger buckets for easier graph readability

In [ ]:
import re
import pandas as pd

desired_groups = [
    "< $99,999",
    "$100,000 - $199,999",
    "$200,000 - $299,999",
    "$300,000 to $399,999",
    "$400,000 to $499,999",
    "$500,000 to $749,999",
    "$750,000 to $999,999",
    "$1,000,000 - $1,999,999",
    "$2,000,000 or More",
]

# --- helper functions ---
def _nums(label):
    if not isinstance(label, str):
        return []
    return [int(x.replace(",", "")) for x in re.findall(r"\$?\s*([\d,]+)", label)]

def to_group(label):
    if not isinstance(label, str):
        return None
    s = label.lower().strip()
    ns = _nums(label)

    if "less" in s or "under" in s or "below" in s:
        return "< $99,999"
    if "or more" in s or "+" in s or "and over" in s:
        if ns and ns[0] >= 2_000_000:
            return "$2,000,000 or More"
        if ns and 1_000_000 <= ns[0] < 2_000_000:
            return "$1,000,000 - $1,999,999"

    lo = ns[0] if ns else None
    hi = ns[1] if len(ns) > 1 else lo

    if lo is None:
        return None
    if (hi or lo) >= 2_000_000:           return "$2,000,000 or More"
    if 1_000_000 <= lo < 2_000_000:       return "$1,000,000 - $1,999,999"
    if 750_000   <= lo <= 999_999:        return "$750,000 - $999,999"
    if 500_000   <= lo <= 749_999:        return "$500,000 to $749,999"
    if 400_000   <= lo <= 499_999:        return "$400,000 to $499,999"
    if 300_000   <= lo <= 399_999:        return "$300,000 to $399,999"
    if 200_000   <= lo <= 299_999:        return "$200,000 - $299,999"
    if 100_000   <= lo <= 199_999:        return "$100,000 - $199,999"
    if lo < 100_000:                      return "< $99,999"
    return None

# --- create the grouped column ---
prop_val["bucket_group"] = prop_val["bucket"].apply(to_group)

# --- drop the original bucket column (clean up) ---
prop_val = prop_val.drop(columns=["bucket"])

# --- aggregate counts by new group ---
agg = (prop_val.dropna(subset=["bucket_group"])
       .groupby(["region", "year", "bucket_group"], as_index=False)
       .agg(Count=("#Properties", "sum")))

# --- recompute share per region-year ---
agg["share"] = agg["Count"] / agg.groupby(["region", "year"])["Count"].transform("sum")

# --- ordering for legend/stack ---
order_map = {b: i for i, b in enumerate(desired_groups)}
agg["bucket_order"] = agg["bucket_group"].map(order_map)


In [ ]:
prop_val.head()

In [ ]:
prop_val.tail()

In [ ]:
prop_val.shape

#### Plot Chart

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path
import re

# ---------- load & prep (same as before) ----------
prop_val_path = Path('../data/austin_property_value.csv')
prop_val = pd.read_csv(prop_val_path)

prop_val = prop_val.rename(columns={
    'Value Bucket': 'bucket',
    'Year': 'year',
    'Place': 'place',
    'share': 'share'
})
prop_val = prop_val[prop_val['place'].isin(['Austin, TX','United States'])].copy()
prop_val['region'] = prop_val['place'].replace({'Austin, TX':'Austin','United States':'U.S.'})
prop_val['year'] = pd.to_numeric(prop_val['year'], errors='coerce').dropna().astype(int)

# ---------- map fine buckets -> 9 big buckets ----------
desired_groups = [
    "< $99,999",
    "$100,000 - $199,999",
    "$200,000 - $299,999",
    "$300,000 to $399,999",
    "$400,000 to $499,999",
    "$500,000 to $749,999",
    "$750,000 to $999,999",
    "$1,000,000 - $1,999,999",
    "$2,000,000 or More",
]
order_map = {b:i for i,b in enumerate(desired_groups)}

def _nums(s):
    if not isinstance(s,str): return []
    return [int(x.replace(',','')) for x in re.findall(r"\$?\s*([\d,]+)", s)]

def to_group(label):
    if not isinstance(label,str): return None
    s = label.lower().strip()
    ns = _nums(label)
    if 'less' in s or 'under' in s or 'below' in s: return "< $99,999"
    if 'or more' in s or '+' in s or 'and over' in s:
        if ns and ns[0] >= 2_000_000: return "$2,000,000 or More"
        if ns and 1_000_000 <= ns[0] < 2_000_000: return "$1,000,000 - $1,999,999"
    lo = ns[0] if ns else None
    hi = ns[1] if len(ns)>1 else lo
    if lo is None: return None
    if (hi or lo) >= 2_000_000:         return "$2,000,000 or More"
    if 1_000_000 <= lo < 2_000_000:     return "$1,000,000 - $1,999,999"
    if 750_000   <= lo <= 999_999:      return "$750,000 - $999,999"
    if 500_000   <= lo <= 749_999:      return "$500,000 to $749,999"
    if 400_000   <= lo <= 499_999:      return "$400,000 to $499,999"
    if 300_000   <= lo <= 399_999:      return "$300,000 to $399,999"
    if 200_000   <= lo <= 299_999:      return "$200,000 - $299,999"
    if 100_000   <= lo <= 199_999:      return "$100,000 - $199,999"
    if lo < 100_000:                    return "< $99,999"
    return None

prop_val['bucket_group'] = prop_val['bucket'].apply(to_group)
prop_val = prop_val.dropna(subset=['bucket_group']).copy()

# ---------- build clean counts only (one row per region-year-bucket_group) ----------
count_col = next((c for c in ['#Properties','Property Value by Bucket','Count','count','n']
                  if c in prop_val.columns), None)
if not count_col:
    raise ValueError("No count column found (e.g., '#Properties').")

prop_val[count_col] = pd.to_numeric(prop_val[count_col], errors='coerce').fillna(0)

agg_counts = (prop_val.groupby(['region','year','bucket_group'], as_index=False)
                        .agg(Count=(count_col,'sum')))
agg_counts['bucket_order'] = agg_counts['bucket_group'].map(order_map)

# ---------- year slider + stacked (normalize from counts) ----------
min_year, max_year = int(agg_counts['year'].min()), int(agg_counts['year'].max())

use_point = hasattr(alt,'selection_point')
if use_point:  # Altair v5
    year_sel = alt.selection_point(fields=['year'],
                                   bind=alt.binding_range(min=min_year, max=max_year, step=1),
                                   value={'year': min_year})
    base = alt.Chart(agg_counts).add_params(year_sel).transform_filter(year_sel)
else:          # Altair v4
    year_sel = alt.selection_single(fields=['year'],
                                    bind=alt.binding_range(min=min_year, max=max_year, step=1),
                                    init={'year': min_year})
    base = alt.Chart(agg_counts).add_selection(year_sel).transform_filter(year_sel)

palette9 = ["#1b9e77","#66a61e","#a6d854","#7570b3","#80b1d3",
            "#e6ab02","#ffd92f","#e7298a","#d95f02"]

chart = base.mark_bar().encode(
    x=alt.X('region:N', title='Region', sort=['Austin','U.S.'], axis=alt.Axis(labelAngle=0)),
    y=alt.Y('sum(Count):Q',                  # ← aggregate counts here
            stack='normalize',
            axis=alt.Axis(title='Share of Homes', format='%'),
            scale=alt.Scale(domain=[0,1], nice=False, zero=True)),  # ← force 0..1
    color=alt.Color('bucket_group:N',
                    title='Home Price Ranges',
                    scale=alt.Scale(domain=desired_groups, range=palette9)),
    order=alt.Order('bucket_order:Q'),
    tooltip=[
        alt.Tooltip('year:O', title='Year'),
        alt.Tooltip('region:N', title='Region'),
        alt.Tooltip('bucket_group:N', title='Price Range'),
        alt.Tooltip('sum(Count):Q', title='Count', format=',')
    ]
).properties(width=520, height=360,
             title='Share of Homes by Price Range — Austin vs. U.S. (Use slider)')

chart


In [ ]:
print(sorted(agg_counts['region'].unique()))  # should be ['Austin','U.S.']
print(agg_counts.duplicated(['region','year','bucket_group']).sum())  # should be 0
